# 01 — EDA & Preprocessing
# Tasks 1 + 2

**Agenda:**
1. Fetch data for all 5 stocks via yfinance
2. Forward-fill missing values
3. Price + volume plots
4. STL decomposition (trend direction)
5. Rolling 30-day std dev (volatility profile)
6. ADF test per stock
7. Log returns
8. Correlation heatmap
9. Train/Test split + MinMaxScaler
10. Save processed CSVs

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import MinMaxScaler
import config

## 1. Fetch Data

In [ ]:
raw_dir = config.DATA_DIR / 'raw'
raw_dir.mkdir(exist_ok=True)

dfs = {}
for sector, ticker in config.TICKERS.items():
    df = yf.download(ticker, start=config.START_DATE, end=config.END_DATE, interval=config.INTERVAL)
    df.columns = df.columns.get_level_values(0)
    df.to_csv(raw_dir / f'{ticker}.csv')
    dfs[sector] = df
    print(f'{sector}: {len(df)} rows, {df.index[0].date()} → {df.index[-1].date()}')

## 2. Forward-Fill + Combine

In [ ]:
close = pd.DataFrame({s: df['Close'].ffill() for s, df in dfs.items()})
volume = pd.DataFrame({s: df['Volume'].ffill() for s, df in dfs.items()})
print(f'Combined close shape: {close.shape}, Missing values: {close.isna().sum().sum()}')

## 3. Price + Volume Plots

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 16))
for i, (sector, df) in enumerate(dfs.items()):
    df['Close'].plot(ax=axes[i, 0], title=f'{sector} — Close Price')
    df['Volume'].plot(ax=axes[i, 1], title=f'{sector} — Volume', color='orange')
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'price_volume_all.png', dpi=150)
plt.show()

## 4. STL Decomposition (trend direction per stock)

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(16, 18))
for i, sector in enumerate(config.TICKERS):
    series = close[sector]
    stl = STL(series, period=252)
    result = stl.fit()
    result.observed.plot(ax=axes[i, 0], title=f'{sector} — Observed')
    result.trend.plot(ax=axes[i, 1], title=f'{sector} — Trend')
    result.seasonal.plot(ax=axes[i, 2], title=f'{sector} — Seasonal')
plt.tight_layout()
plt.savefig(config.PLOTS_DIR / 'stl_decomposition_all.png', dpi=150)
plt.show()

## 5. Rolling 30-Day Std Dev

In [ ]:
roll_std = close.rolling(30).std()
roll_std.plot(figsize=(14, 6), title='Rolling 30-Day Std Dev — All Stocks')
plt.savefig(config.PLOTS_DIR / 'rolling_std.png', dpi=150)
plt.show()

## 6. ADF Test

In [ ]:
adf_results = []
for sector in config.TICKERS:
    series = close[sector]
    result = adfuller(series.dropna())
    adf_results.append({
        'Sector': sector,
        'ADF Statistic': round(result[0], 4),
        'p-value': round(result[1], 6),
        'Critical Values': result[4],
        'Stationary (p<0.05)': 'Yes' if result[1] < 0.05 else 'No'
    })
adf_df = pd.DataFrame(adf_results)
print(adf_df.to_string(index=False))

## 7. Log Returns

In [ ]:
log_returns = np.log(close / close.shift(1)).dropna()
print(f'Log returns shape: {log_returns.shape}')
log_returns.to_csv(config.DATA_DIR / 'processed' / 'log_returns.csv')

## 8. Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(log_returns.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Log Returns Correlation — All Stocks')
plt.savefig(config.PLOTS_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

## 9. Train/Test Split + Scaler

In [ ]:
train = close.loc[:config.TRAIN_END]
test  = close.loc[config.TEST_START:config.TEST_END]

print(f'Train: {train.index[0].date()} → {train.index[-1].date()}, {len(train)} rows')
print(f'Test:  {test.index[0].date()}  → {test.index[-1].date()},  {len(test)} rows')

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(train),
    index=train.index,
    columns=train.columns
)
test_scaled = pd.DataFrame(
    scaler.transform(test),
    index=test.index,
    columns=test.columns
)

import pickle
with open(config.DATA_DIR / 'processed' / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Scaler saved.')

## 10. Save Processed Data

In [ ]:
processed_dir = config.DATA_DIR / 'processed'
processed_dir.mkdir(exist_ok=True, parents=True)

train_scaled.to_csv(processed_dir / 'train_scaled.csv')
test_scaled.to_csv(processed_dir / 'test_scaled.csv')
close.to_csv(processed_dir / 'close_prices.csv')

print('All processed files saved to data/processed/')